# exp03: efecto aislado de mejoras sobre un modelo base fijo
Modelo base: CNN residual pequeña en la grilla de 3 km. Entrada: GFS interpolado (t, t-1, t-2), DEM, DEM - DEM suavizado, hora seno/coseno.
Objetivo: WRF - GFS(t) interpolado. Pérdida: promedio_v(MSE_v / REF_v^2). Validación temporal: times >= 2024061600.
Variantes: (1) contexto t-6, t-12, t-24 por timestamps; (2) rasgos derivados (rapidez, diferencias t-(t-1), punto de rocío); (3) recorte de RH a [0,100] en la salida; (4) promedio de semillas.

In [1]:
import time, os
from datetime import datetime, timedelta
import numpy as np, pandas as pd, torch, torch.nn as nn
torch.set_num_threads(2)
T0 = time.time()
DATA = '../data'
REF = np.array([25.360, 2.0652, 2.6776, 2.0876], dtype=np.float32)
VARS = ['RH', 'T', 'U', 'V']
EPOCHS = 4
tr = np.load(f'{DATA}/train.npz'); st = np.load(f'{DATA}/static.npz')
X = tr['X'].astype(np.float32); Y = tr['Y'].astype(np.float32); times = tr['times'].astype(str)
print(X.shape, Y.shape, times[:3])

(2935, 17, 13, 4) (2935, 70, 35, 4) ['2024021800' '2024021801' '2024021802']


In [2]:
# Interpolación bilineal separable GFS -> WRF (coordenadas reales)
def interp_matrix(src, dst):
    order = np.argsort(src); s = src[order]
    W = np.zeros((len(dst), len(src)), np.float32)
    for i, d in enumerate(dst):
        j = np.clip(np.searchsorted(s, d) - 1, 0, len(s) - 2)
        w = np.clip((d - s[j]) / (s[j + 1] - s[j]), 0, 1)
        W[i, order[j]] += 1 - w; W[i, order[j + 1]] += w
    return W
Wlat = interp_matrix(st['gfs_lats'], st['wrf_lats']); Wlon = interp_matrix(st['gfs_lons'], st['wrf_lons'])
G = np.einsum('ia,nabc,jb->ncij', Wlat, X, Wlon).astype(np.float32)  # (N,4,70,35)
print('G', G.shape)

G (2935, 4, 70, 35)


In [3]:
# Índices de rezago por timestamp (si falta la hora, se usa el rezago más cercano disponible hacia t)
dts = [datetime.strptime(t, '%Y%m%d%H') for t in times]
pos = {d: i for i, d in enumerate(dts)}
def lag_idx(L):
    out = np.empty(len(dts), int)
    for i, d in enumerate(dts):
        k = i
        for l in range(L, 0, -1):
            j = pos.get(d - timedelta(hours=l))
            if j is not None: k = j; break
        out[i] = k
    return out
LAGS = {L: lag_idx(L) for L in [1, 2, 6, 12, 24]}
for L in [1, 2, 6, 12, 24]:
    print('rezago', L, 'h exacto:', np.mean([pos.get(d - timedelta(hours=L)) is not None for d in dts]).round(3))

rezago 1 h exacto: 0.995
rezago 2 h exacto: 0.992
rezago 6 h exacto: 0.984
rezago 12 h exacto: 0.974
rezago 24 h exacto: 0.956


In [4]:
# Estáticos
dem = st['dem'].astype(np.float32)
k = np.ones(9, np.float32) / 9
sm = dem.copy()
for _ in range(3):
    p = np.pad(sm, 4, mode='edge')
    sm = np.apply_along_axis(lambda r: np.convolve(r, k, 'valid'), 1, p[4:-4, :])
    p = np.pad(sm, 4, mode='edge')
    sm = np.apply_along_axis(lambda c: np.convolve(c, k, 'valid'), 0, p[:, 4:-4])
STATIC = np.stack([(dem - dem.mean()) / dem.std(), (dem - sm) / (dem - sm).std()])
hours = np.array([d.hour for d in dts])
HSC = np.stack([np.sin(2 * np.pi * hours / 24), np.cos(2 * np.pi * hours / 24)], 1).astype(np.float32)

def dewpoint(Tk, RH):
    Tc = Tk - 273.15; rh = np.clip(RH, 1, 100)
    a, b = 17.625, 243.04
    g = np.log(rh / 100) + a * Tc / (b + Tc)
    return b * g / (a - g)

def build(variant):
    feats = [G, G[LAGS[1]], G[LAGS[2]]]
    if variant == 'temporal':
        feats += [G[LAGS[6]], G[LAGS[12]], G[LAGS[24]]]
    if variant == 'derivados':
        spd = np.sqrt(G[:, 2] ** 2 + G[:, 3] ** 2)[:, None]
        diff = G - G[LAGS[1]]
        td = dewpoint(G[:, 1], G[:, 0])[:, None]
        dep = (G[:, 1] - 273.15)[:, None] - td
        feats += [spd, diff, td, dep]
    return np.concatenate(feats, 1).astype(np.float32)

target = ((Y.transpose(0, 3, 1, 2) - G) / REF[None, :, None, None]).astype(np.float32)
val = np.array([t >= '2024061600' for t in times]); trn = ~val
print('train', trn.sum(), 'val', val.sum())

train 2599 val 336


In [5]:
# Modelo
class Res(nn.Module):
    def __init__(s, c):
        super().__init__(); s.a = nn.Conv2d(c, c, 3, padding=1); s.b = nn.Conv2d(c, c, 3, padding=1); s.r = nn.GELU()
    def forward(s, x): return x + s.b(s.r(s.a(s.r(x))))
class Net(nn.Module):
    def __init__(s, cin, c=16, nb=3):
        super().__init__()
        s.inp = nn.Conv2d(cin, c, 3, padding=1); s.body = nn.Sequential(*[Res(c) for _ in range(nb)]); s.out = nn.Conv2d(c, 4, 1)
    def forward(s, x): return s.out(s.body(s.inp(x)))

def run(variant, seed=0):
    F = build(variant)
    mu = F[trn].mean((0, 2, 3), keepdims=True); sd = F[trn].std((0, 2, 3), keepdims=True) + 1e-6
    F = (F - mu) / sd
    n = len(F)
    full = np.concatenate([F, np.broadcast_to(STATIC, (n, 2, 70, 35)),
                           np.broadcast_to(HSC[:, :, None, None], (n, 2, 70, 35))], 1).astype(np.float32)
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net(full.shape[1]); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=1e-4)
    xt = torch.from_numpy(full[trn]); yt = torch.from_numpy(target[trn])
    steps = EPOCHS * int(np.ceil(len(xt) / 32)); sch = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=steps)
    t0 = time.time()
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(len(xt))
        for i in range(0, len(xt), 32):
            b = perm[i:i + 32]
            loss = ((net(xt[b]) - yt[b]) ** 2).mean()  # objetivo ya dividido por REF -> promedio_v MSE_v/REF_v^2
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    net.eval()
    with torch.no_grad():
        p = torch.cat([net(torch.from_numpy(full[val][i:i + 64])) for i in range(0, val.sum(), 64)]).numpy()
    pred = p * REF[None, :, None, None] + G[val]
    print(f'{variant} seed{seed}: {time.time() - t0:.0f}s')
    return pred

Yv = Y[val].transpose(0, 3, 1, 2)
def metrics(pred):
    rm = np.sqrt(((pred - Yv) ** 2).mean((0, 2, 3)))
    return 1 - np.mean(rm / REF), rm

In [6]:
# Referencias sin entrenar
rows = []
def add(name, pred, note=''):
    s, rm = metrics(pred)
    rows.append(dict(variante=name, score_local=s, **{f'RMSE_{v}': r for v, r in zip(VARS, rm)}, nota=note))
    print(name, round(s, 4), rm.round(3))
add('gfs_interpolado (sin modelo)', G[val], 'referencia')

gfs_interpolado (sin modelo) -0.1637 [15.439  3.759  2.553  2.656]


In [7]:
# Base y variantes
P = {}
P['base'] = run('base', 0); add('base', P['base'], 't,t-1,t-2 + DEM + DEM-suavizado + hora')
P['temporal'] = run('temporal', 0); add('temporal t-6,t-12,t-24', P['temporal'], '+12 canales por timestamps')
P['derivados'] = run('derivados', 0); add('derivados GFS', P['derivados'], '+rapidez, diff t-(t-1), Td, T-Td')
clip = P['base'].copy(); clip[:, 0] = np.clip(clip[:, 0], 0, 100)
add('base + recorte RH [0,100]', clip, 'postproceso, sin reentrenar')
print('fraccion RH fuera de [0,100] en base:', np.mean((P['base'][:, 0] < 0) | (P['base'][:, 0] > 100)).round(4))
print('minutos', (time.time() - T0) / 60)

base seed0: 506s


base 0.1841 

[12.2    1.888  2.301  2.106]


temporal seed0: 628s


temporal t-6,t-12,t-24 0.1987 

[12.409  1.818  2.247  2.081]


derivados seed0: 163s


derivados GFS 0.2006 [12.164  1.858  2.239  2.05 ]
base + recorte RH [0,100] 0.1842 [12.189  1.888  2.301  2.106]
fraccion RH fuera de [0,100] en base: 0.0053
minutos 27.59164969921112


In [8]:
# Semillas (si alcanza el tiempo)
if datetime.now().strftime('%H%M') < '1840':
    P['s1'] = run('base', 1); add('base semilla 1', P['s1'], 'individual')
    P['s2'] = run('base', 2); add('base semilla 2', P['s2'], 'individual')
    add('promedio 3 semillas (base)', (P['base'] + P['s1'] + P['s2']) / 3, 'ensamble')
else:
    print('sin tiempo para semillas')

sin tiempo para semillas


In [9]:
# Tabla final
df = pd.DataFrame(rows)
b = df.loc[df.variante == 'base', 'score_local'].values[0]
df['dif_vs_base'] = df.score_local - b
os.makedirs('results', exist_ok=True)
df.to_csv('results/exp03_resultados.csv', index=False)
pd.set_option('display.width', 200)
print(df.round(4).to_string(index=False))
print('minutos totales', (time.time() - T0) / 60)

                    variante  score_local  RMSE_RH  RMSE_T  RMSE_U  RMSE_V                                   nota  dif_vs_base
gfs_interpolado (sin modelo)      -0.1637  15.4390  3.7592  2.5533  2.6559                             referencia      -0.3478
                        base       0.1841  12.1997  1.8883  2.3008  2.1059 t,t-1,t-2 + DEM + DEM-suavizado + hora       0.0000
      temporal t-6,t-12,t-24       0.1987  12.4086  1.8177  2.2467  2.0805             +12 canales por timestamps       0.0146
               derivados GFS       0.2006  12.1641  1.8579  2.2385  2.0505       +rapidez, diff t-(t-1), Td, T-Td       0.0165
   base + recorte RH [0,100]       0.1842  12.1894  1.8883  2.3008  2.1059            postproceso, sin reentrenar       0.0001
minutos totales 27.768611272176106
